## Week 2 Day 2 - Orchestration

Our first Agentic Framework project!!

### Part 1: Email Setup

### Part 2: Orchestrating by code

### Part 3: Orchestrating by LLMs

- 3a: via Tools
- 3b: via Handoffs

## Part 1: Email Setup

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">IMPORTANT PLEASE READ - Sending Emails</h2>
            <span style="color:#ff7800;">We are going to write an agent that sends emails. The best way to do this is with an email provider
            like SendGrid or Resend. But this requires a fair amount of heavy-lifting: you need to send from a proper email host where you own the domain and you can prove it with DNS records. It's quite a hassle.<br/><br/>
            So we're going to do it the free & simple way: using SMTP server configuration to send direct from your email. It's an easy setup, but it's not as powerful (eg you can't receive emails). If you'd like to take this further, use SendGrid or Resend instead..<br/><br/>
            And actually the email send is optional. We're doing this to demonstrate an Agent sending emails. The important part is the Agent work, not the email sending. Feel free to swap out the function for a Pushover push if you'd prefer.
            </span>
        </td>
    </tr>
</table>

## Setting up to send emails from your SMTP server

### STEP 1: Determining your SMTP Server

Either google or ask ChatGPT / Claude for the SMTP server for your email. Here are some common ones. Some email providers might not have SMTP servers enabled (eg Microsoft 365 for work/school).

Google: smtp.gmail.com  
Outlook.com / Hotmail / Live: smtp-mail.outlook.com  
Microsoft 365: smtp.office365.com  
iCloud Mail: smtp.mail.me.com  

Add to your .env file:  

`EMAIL_SMTP_SERVER=xxxx`

### STEP 2: Obtain an app specific password

Google how to do this for your email provider. For gmail, you need to have 2-step verification on. Then visit this page:

https://myaccount.google.com/apppasswords

Give it any name; copy the password and add it to your .env file, removing the spaces that it adds. (It should be 16 characters, no spaces.)

`EMAIL_APP_PASSWORD=xxxx`

### STEP 3: Add in your email address:

`EMAIL_ADDRESS=xxx`

Remember to Save the .env file!

In [1]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
import smtplib
from email.message import EmailMessage

load_dotenv(override=True)
ModelName = "gpt-5.4-mini"

In [2]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

In [3]:
def SendEmail(Subject: str, TextBody: str, HTMLBody: str):
    """Send an email"""
    Message = EmailMessage()
    Message["From"] = EMAIL_ADDRESS
    Message["To"] = EMAIL_ADDRESS
    Message["Subject"] = Subject
    Message.set_content(TextBody)
    Message.add_alternative(HTMLBody, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as Server:
        Server.starttls()
        Server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        Server.send_message(Message)

In [4]:
SendEmail("Progress Message", "Fingers crossed..", "<html><body><strong>Fingers</strong> crossed..</body></html>")

## Agent Orchestration

There are 2 models for Agent Orchestration; by code and by LLMs.

By code: more predictable and deterministic.

By LLMs: more powerful.

An excellent write-up is here:

https://openai.github.io/openai-agents-python/multi_agent/

We will start with by Code.

## Part 2: Orchestrating by Code

In [5]:
ModelName = "gpt-4o-mini"

CommonIntro = """
              You are a sales agent working for ComplAI, a company that provides a SaaS tool
              for ensuring SOC2 compliance and preparing for audits, powered by AI. You
              write cold sales emails.
              """

Instructions1 = CommonIntro + "You write in a professional, serious style with gravitas"
Instructions2 = CommonIntro + "You write witty, engaging and humorous emails likely to get a response"
Instructions3 = CommonIntro + "You write concise, to the point emails in the style of a busy executive"

SalesAgent1 = Agent(name="Professonal Sales Agent", instructions=Instructions1, model=ModelName)
SalesAgent2 = Agent(name="Witty Sales Agent", instructions=Instructions2, model=ModelName)
SalesAgent3 = Agent(name="Executive Sales Agent", instructions=Instructions3, model=ModelName)

In [6]:
Result = Runner.run_streamed(SalesAgent1, input="Write a cold sales email")
async for event in Result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print (event.data.delta, end="", flush=True)

Subject: Elevate Your SOC 2 Compliance and Audit Preparedness

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I represent ComplAI, a pioneering SaaS solution designed to streamline and fortify your SOC 2 compliance journey.

In today’s regulatory environment, ensuring robust compliance is not just a checkbox exercise; it is imperative for building trust with clients and stakeholders. Our AI-powered platform not only simplifies the compliance process but also enhances your preparation for audits, making it more efficient and less burdensome.

With ComplAI, organizations like yours can expect:

1. **Automated Compliance Monitoring**: Stay ahead of potential issues with real-time insights and automated audit trails.
  
2. **Comprehensive Documentation**: Effortlessly generate and manage the necessary documentation to meet SOC 2 requirements.

3. **Expert Guidance**: Benefit from best practices and AI-driven recommendations tailored specifically t

In [10]:
import sendgrid
from sendgrid.helpers.mail import Email, Mail, To, Content

@function_tool
def SendEmail(Subject: str, HTMLBody: str):
    """ Send an email to a prospective client """
    SG = sendgrid.SendGridAPIClient(api_key = os.getenv("SENDGRID_API_KEY"))
    FromEmail = Email("spraybubble@gmail.com")
    ToEmail = To("spraybubble@gmail.com")
    content = Content("text/html",HTMLBody)
    mail = Mail(FromEmail, ToEmail, Subject, content).get()
    SG.client.mail.send.post(request_body=mail)
    return {"Mail sent":"OK"}

In [8]:
## Have each of the 3 agents write a mail. Then use another LLM 
## to evaluate which is the best email and use the email tool to send the mail

Message = "Write a cold sales email"

DecisionInstructions = """
                       You are a sales manager working for ComplAI, a company that provides a SaaS tool
                       for ensuring SOC2 compliance and preparing for audits, powered by AI. You have to
                       send a cold email to a sales prospect.
                       You are given drafts of 3 mails written by 3 different agents. You must decide 
                       which of these 3 mails is the most effectivem and likely to get a response from
                       the recipient
                       Please note that you have to select ONLY ONE email. Also, do not draft the email yourself.
                       You must choose one of the 3 email drafts provided to you
                       Once you have selected the mail to be sent, craft a subject line for it and convert
                       the email body into well-formatted, highly engaging HTML format.
                       Use the SendEmail tool to send the message. 
                       """

with trace("Sales Manager mail"):
    Results = await asyncio.gather(
        Runner.run(SalesAgent1, Message),
        Runner.run(SalesAgent2, Message),
        Runner.run(SalesAgent3, Message)
    )

    Outputs = [Result.final_output for Result in Results]
    print ("\n\n".join(Outputs))

    CombinedEmails = "## Set of cold email drafts: \n\n" + "\n## Email Draft \n\n".join(Outputs)

    SalesManager = Agent(name="Sales Manager", instructions=DecisionInstructions, model="gpt-5.4-mini", 
                              tools=[SendEmail], model_settings=ModelSettings(tool_choice="required"))
    Result = await Runner.run(SalesManager, CombinedEmails)
    print (Result)



Subject: Streamline Your SOC2 Compliance Process with ComplAI

Dear [Recipient's Name],

I hope this message finds you well. 

In today’s regulatory landscape, achieving SOC2 compliance is not only essential for demonstrating your commitment to data security but also crucial for gaining the trust of your clients. However, the traditional methods of preparing for compliance audits can often be cumbersome, time-consuming, and prone to human error.

At ComplAI, we have developed an AI-powered SaaS tool specifically designed to simplify the SOC2 compliance process. Our platform offers robust solutions that enable your organization to:

- Automate compliance tasks, significantly reducing the time spent on manual work.
- Maintain real-time monitoring of compliance status and potential risks.
- Generate comprehensive reports with ease, ensuring your audit readiness at all times.

Many companies like yours have reported improved efficiency and reduced audit preparation time by over 30% after i

Now go and check out the trace:

https://platform.openai.com/traces

## Part 3: Orchestrating by LLMs

### 3a: via Tools

The simplest way to have 1 Agent choose to invoke another is by treating it as a tool call.

The OpenAI Agents SDK gives a very simple way to do this.

This works best when the flow is:

Agent A -> Agent B -> Agent A

And for the classic "Planning Agent" situation.

In [11]:
ToolDesc = "Use this tool to write a cold sales email. In the input, just instruct it to generate the body for a sales email"
Tool1 = SalesAgent1.as_tool(tool_name="SalesAgent1", tool_description=ToolDesc)
Tool2 = SalesAgent2.as_tool(tool_name="SalesAgent2", tool_description=ToolDesc)
Tool3 = SalesAgent3.as_tool(tool_name="SalesAgent3", tool_description=ToolDesc)

SalesMgrInstructions = """You are a sales manager working for ComplAI, a company that provides a SaaS tool
                       for ensuring SOC2 compliance and preparing for audits, powered by AI. You have to
                       send a cold email to a sales prospect."""

Task = """
       You have to write a cold sales email to a prospect. Please follow the below steps:

       1. Generate Drafts: Use the 3 SalesAgent tools to generate a draft email each. Do not
       proceed till you have the 3 drafts ready.
       2. Evaluate and Select: Review the three draft emails, and select the best email based
       on your judgement of which email is most likely to elicit a response.
       3. Write a subject line for the email and convert the email body to a crisply formatted HTML body
       4. Use the SendEmail tool to send the email to the prospect

       Crucial Rules:
       - Do not write the email yourself. You must select one of the draft emails written by the SalesAgent tools
       - You can select only ONE email for sending. Do not select more than one email
       - The subject line of the mail has to be written by you, and it can differ from what the SalesAgent tools provide
       - The email body should be properly HTML-formatted prior to sending to the SendEmail tool
       """

Tools = [Tool1, Tool2, Tool3, SendEmail]

SalesManager = Agent(name="SalesManagerwithTools", instructions = SalesMgrInstructions, model="gpt-5.4-mini",
                     tools = Tools, model_settings = ModelSettings(tool_choice="required"))

with trace("Sales Manager with Agent Tools"):
       Result = await Runner.run(SalesManager, Task)
       print (Result)



RunResult:
- Last agent: Agent(name="SalesManagerwithTools", ...)
- Final output (str):
    Done — I selected the strongest draft, wrote a fresh subject line, formatted the body in HTML, and sent the email.
- 10 new item(s)
- 3 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [ ]:
# draw_graph(SalesManager)

## Remember to check the trace

https://platform.openai.com/traces

And then check your email!! Also look in your Junk / Spam folder - after all, this is basically a spam message..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Unreliable results?</h2>
            <span style="color:#ff7800;">This is par for the course with Agentic AI, and particularly with Orchestrating by LLM.
            To resolve it, you'll need to experiment and iterate with the prompts. Particularly with smaller models, more experiments
            may be needed to get reliable outcomes.
            </span>
        </td>
    </tr>
</table>

## Part 3: Orchestrating by LLMs

### 3a: via Handoffs

I am not a fan of handoffs. They seem very unreliable. They're not used consistently by other frameworks.

Behind the scenes, OpenAI Agents SDK has implemented these with Tools anyway.

### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

A -> B -> A

With handoffs, control passes across

A -> B

In [14]:
EmailMgrInstructions = """
                       You are provided with a text email body. You must generate a subject line
                       for the email, and format the text email body into a crisply formatted, 
                       easy-to-read HTML email body.
                       Then use the SendEmail tool to send one and only one email to the prospect.
                       You must use the SendEmail tool to send the email before you finish.
                       """

EmailManager = Agent(name="EmailManager", instructions=EmailMgrInstructions, model="gpt-5.4-mini",
                    tools = [SendEmail], handoff_description="Write subject line, convert to HTML and send email",
                    model_settings = ModelSettings(tool_choice="required"))

SalesMgrInstructions = """You are a sales manager working for ComplAI, a company that provides a SaaS tool
                       for ensuring SOC2 compliance and preparing for audits, powered by AI."""

Task = """
       You have to write a cold sales email to a prospect. Please follow the below steps:

       1. Generate Drafts: Use the 3 SalesAgent tools to generate a draft email each. Do not
       proceed till you have the 3 drafts ready.
       2. Evaluate and Select: Review the three draft emails, and select the best email based
       on your judgement of which email is most likely to elicit a response.
       3. Send the body of the best email draft to the EmailManager agent. Hand it off to the 
       Email Manager agent who will be responsible for formatting and sending the email.

       Crucial Rules:
       - Do not write the email yourself. You must select one of the draft emails written by the SalesAgent tools
       - You can select only ONE email for handing off to the EmailManager agent. Do not select more than one email
       """

Tools = [Tool1, Tool2, Tool3]

SalesManager = Agent(name="SalesManagerwithTools", instructions = SalesMgrInstructions, model="gpt-5.4-mini",
                     tools = Tools, model_settings = ModelSettings(tool_choice="required"), handoffs=[EmailManager])

with trace("Agent Handoffs"):
    Result = await Runner.run(SalesManager,Task)

In [ ]:
# draw_graph(sales_manager)

### Remember to check the trace

https://platform.openai.com/traces

And then check your email!!

Note that handoffs can be unrealiable and a little bit frustrating. I needed to force the tool use otherwise this didn't work. If you don't get reliable behavior, try iterating on the prompts - or use a larger model. And enjoy the process; this is what Agentic AI is all about!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Try extending with some more orchestration - perhaps another iteration of the email to refine it. 
            Use Orchestrating via Code and Orchestrating via LLMs.  
            HARD CHALLENGE: switch to using a professional email provider like SendGrid or Resend. Then handle the user replying. 
            Then have the SDR respond to keep the conversation going!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">This is immediately applicable to Sales Automation; but more generally this could be applied to  end-to-end automation of any business process through conversations and tools. Think of ways you could apply an Agent solution
            like this in your day job.
            </span>
        </td>
    </tr>
</table>